In [15]:
import json
import os
from urllib.request import Request, urlopen

from dotenv import load_dotenv
from langchain_core.callbacks import CallbackManagerForLLMRun
from langchain_core.language_models.llms import LLM

In [16]:
load_dotenv()  # Load environment variables from .env file

True

In [17]:
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
    raise RuntimeError("Set OPENROUTER_API_KEY in your .env file.")

In [18]:
class OpenRouterTextLLM(LLM):
    model_name: str
    max_tokens: int = 1024
    api_url: str = "https://openrouter.ai/api/v1/chat/completions"

    @property
    def _llm_type(self) -> str:
        return "openrouter-text-completion"

    def _call(
        self,
        prompt: str,
        stop: list[str] | None = None,
        run_manager: CallbackManagerForLLMRun | None = None,
        **kwargs,
    ) -> str:
        api_key = os.getenv("OPENROUTER_API_KEY")
        if not api_key:
            raise RuntimeError("Set OPENROUTER_API_KEY in your .env file.")

        payload = {
            "model": self.model_name,
            "prompt": prompt,
            "max_tokens": self.max_tokens,
        }
        if stop:
            payload["stop"] = stop

        request = Request(
            self.api_url,
            data=json.dumps(payload).encode("utf-8"),
            headers={
                "Authorization": f"Bearer {api_key}",
                "Content-Type": "application/json",
            },
            method="POST",
        )
        with urlopen(request, timeout=30) as response:
            result = json.loads(response.read().decode("utf-8"))

        return result["choices"][0]["text"]


llm = OpenRouterTextLLM(model_name="openai/gpt-4o-mini")

In [19]:
response = llm.invoke("What is the capital of France?")
print(response)

The capital of France is Paris.
